# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Adriel Hewlett
**Student ID:** 16092027

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata
API_KEY=userdata.get('API-Key')

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "openai/gpt-oss-120b"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
#It tells me the model from the lab isn't there anymore
models = client.models.list()

for model in models.data:
    print(model.id)
model_name = "llama-3.3-70b-versatile"

try:
    model = client.models.retrieve(model_name)
    print("Available:", model.id)
except Exception as e:
    print("Not available:", e)

qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b
groq/compound
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
groq/compound-mini
openai/gpt-oss-120b
allam-2-7b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
Not available: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


In [4]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",temperature=0.7,
            max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content,response.usage
# TODO: Call it once with a simple question and print the answer.
response,usage=ask_llm("Is a woman a man?")
print(response)
print(usage)
# TODO: Print response.usage as well — how many tokens did your call consume?
#86

No. In everyday usage and most cultural contexts, “woman” and “man” refer to two distinct gender identities:

* **Woman** – typically used for people who identify as female, whether they were assigned female at birth or later come to identify as a woman.
* **Man** – typically used for people who identify as male, whether they were assigned male at birth or later come to identify as a man.

So, under those standard definitions, a woman is not a man.

That said, gender is more complex than a simple binary for many people. Some individuals identify as non‑binary, gender‑fluid, or another gender that doesn’t fit neatly into “woman” or “man.” Additionally, a person’s gender identity may differ from the sex they were assigned at birth (e.g., a transgender woman was assigned male at birth but identifies and lives as a woman).

In short, while a woman is not a man in the conventional sense, respecting each person’s self‑identified gender is the most inclusive and accurate way to talk about gen

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** The system role tells the model what it should do and what kind of criteria it should tailor its responses to, and the user role tells the model the prompt it responds to. The system tells the model "You are a resourceful assistant" and the and the user tells it "Tell me 3 things I should do for more girls to like me". A token is a unit of quantization that represents the encoded value of a fragment of text, and API providers bill per token rather than per request because more tokens proportionally increase the computation demand on a good response.

### Part 1.2 — Temperature: the randomness dial

In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
answers={}
for i in range(10):
  if i<5:
    val=0.0
  else:
    val=1.2
  response,usage=ask_llm("Make a word for wanting to borrow someone's clothes.",temperature=val)
  answers[f"{i}{val}"]=response
for i in answers.keys():
  if i[0]=="0":
    print("Temperature of 0:")
  if i[0]=="5":
   print("Temperature of 1.2:")
  print(f"Answer {int(i[0])+1} : {answers[i]}")
# TODO: Print all 10 answers, grouped by temperature.

Temperature of 0:
Answer 1 : **Word:** **Cloth‑borrowitis** *(noun)*  

**Pronunciation:** /klɔːθ‑ˈbɹoʊ.ɪtɪs/  

**Definition:**  
A strong, often playful desire to temporarily wear someone else’s clothing—whether for the feel of the fabric, the style, or simply the novelty of “trying on” another person’s wardrobe.

**Etymology:**  
- **Cloth** – from Old English *clað* “fabric, garment.”  
- **Borrow** – from Old English *borhwan* “to lend, to take temporarily.”  
- **‑itis** – a suffix borrowed from medical terminology (e.g., *arthritis*, *nostalgia*) that denotes a condition or persistent feeling.  

**Usage examples:**  

1. *“I’ve got a case of cloth‑borrowitis today—can I try on your new jacket?”*  
2. *“Her cloth‑borrowitis was obvious when she kept eye‑bopping at the vintage‑store mannequins.”*  
3. *“He confessed his cloth‑borrowitis to his roommate, who gladly offered his oversized sweater.”*  

**Related forms:**  

- **Cloth‑borrowitic** *(adj.)* – describing someone who is

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** I observed that accross all, the model repeated words; but this happened mainly with the temperature of 0; the temperature of 1.2 used more words. It also changed up the answer structure, while the first 5 kept the same rigid answer structure. I think the higher temperature response would be appropriate since it is less easy to predict and would be more "thoughtful" of its answers.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [7]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT=[]
SUMMARY_PROMPT_V1=f"""Summarize this letter acutely:
{LETTERS["L002"]}
"""
SUMMARY_PROMPT.append(SUMMARY_PROMPT_V1)
response,usage=ask_llm(SUMMARY_PROMPT_V1,temperature=1.2)
print("V1 attempt")
print(response)
SUMMARY_PROMPT_V1=f"""Summarize this letter acutely:
{LETTERS["L006"]}
"""
SUMMARY_PROMPT.append(SUMMARY_PROMPT_V1)
response,usage=ask_llm(SUMMARY_PROMPT_V1,temperature=1.2)
print(response)
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_V1=f"""Summarize this letter acutely:
{LETTERS["L002"]}
"""
system_prompt="You are a microfinance loan officer's assistant. Be neutral, pragmatic yet considerate. Be professional and resourceful. No invented details."
SUMMARY_PROMPT_V2=[system_prompt,SUMMARY_PROMPT_V1,response]
SUMMARY_PROMPT.append(SUMMARY_PROMPT_V2)
response,usage=ask_llm(SUMMARY_PROMPT_V2[1],SUMMARY_PROMPT_V2[0])
print("V2 response")
print(response)
SUMMARY_PROMPT_V1=f"""Summarize this letter acutely:
{LETTERS["L006"]}
"""
SUMMARY_PROMPT_V2=[system_prompt,SUMMARY_PROMPT_V1,response]
SUMMARY_PROMPT.append(SUMMARY_PROMPT_V2)
response,usage=ask_llm(SUMMARY_PROMPT_V2[1],SUMMARY_PROMPT_V2[0])
print(response)
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
#V1 and V2 outputs appear to be similar, but V2 tends to say more and use more professional words.

V1 attempt
Kwame Boateng, a commercial driver in Kumasi, requests an urgent GHS 25,000 loan to repair his trotro engine and cover personal debts. He says business is currently slow but expects improvement after the festive season, has no collateral, and pledges to repay the money once his income resumes. He asks for quick assistance.
**Summary:**  
Kofi, a 22‑year‑old entrepreneur, is requesting a GHS 50,000 loan to launch a car‑washing service, a provision shop, and a phone‑import business from Dubai. He claims strong energy, business acumen (per friends), and promises to repay the loan within a year once the ventures are profitable. He offers no collateral but asserts his trustworthiness.
V2 response
**Summary**

Kwame Boateng, a commercial driver in Kumasi, requests an urgent loan of GHS 25,000 to repair his trotro engine and clear personal debts. He notes that business is currently slow but expects improvement after the festive season. He has no collateral to offer and proposes to 

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** As discussed,V1's output had more informal dialogue and less content compared to V2's. This made it less professionally suitable. There were no other concrete problems; V2 just spoke more. "No invented details" is an important part of the application because it prevents the model from hallucinating(the failure mode).

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [8]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import pandas as pd
import json
letter="""Dear Sir/Madam,
I am Adwoa Asiedu. I work with telecommunications companies like AirtelTigo and Glo. My pay is not
periodic; every task has its pay. I will be travelling a long distance for a specific opportunity, and need a loan of
5,000 GHS to rent the car and cover fuel expenses. Collateral can be coallated."""
fields_prompt="""Return only a JSON object with applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), repayment_months (number or null)."""
response,usage=ask_llm(fields_prompt,letter,temperature=0)
#print(response)
EXTRACT_PROMPT=[]
EXTRACT_PROMPT.append(fields_prompt)
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text,temperature=0):
    response,usage=ask_llm("""Return only a JSON object with applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), repayment_months (number or null).""",letter_text,temperature=temperature)
    response=response.removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(response)
    except:
        print("Parse failure")
        return None
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
fieldsForLetters=pd.DataFrame(columns=["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"])
for i in range(6):
    fieldsForLetters.loc[i]=extract_fields(LETTERS[f"L00{i+1}"])
display(fieldsForLetters)


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,Akosua Mensah,8000,Buy a deep freezer and expand into frozen foods,900,True,20
1,Kwame Boateng,25000,Repair trotro engine and settle personal debts,None,False,None
2,Efua Darko,15000,Purchase two industrial sewing machines and fa...,2800,True,15
3,Yaw Owusu,12000,feed and purchase 500 new layers,1500,True,18
4,Adenta Women's Weaving Cooperative,30000,Buy a bulk order of yarn directly from the fac...,None,True,16
5,Kofi,50000,"car washing business, provision shop, import p...",None,False,12


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** That is because that would be like mixing training and test data sets; it would invalidate testing results. The "use null, do not guess" instruction prevented the model from hallucinating and affected the meaning the model got from the prompt for the better. Temperature=0 is the right choice for extraction because the model repeats almost all the same things on every run, so it can be extracted at any point; but it is horrible for creative tasks exactly for that same reason. It does not show any novelty, so it is a bad pick.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [9]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
def brief_prompt_gen(letter_text,extracted_json):
    extracted_json=json.dumps(extracted_json)
    response,usage=ask_llm("Learn this text and json,make a response you can use as your memory for further prompting.."+letter_text,extracted_json)
    response,usage=ask_llm("""Give me the strengths of the letter in bullet points,risks/red flags in bullet points too,
    missing information the officer should request,a suggested next step (e.g. "invite for interview", "request documents",
    "flag for senior review"); but leave the final decision to humans.Separate each field with +-+.""",response)
    return response
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
BRIEF_PROMPT=[]
response=brief_prompt_gen(LETTERS["L001"],extract_fields(LETTERS["L001"]))
print("First Letter")
print(response)
BRIEF_PROMPT.append(response)
response=brief_prompt_gen(LETTERS["L002"],extract_fields(LETTERS["L002"]))
print("Second Letter")
print(response)
BRIEF_PROMPT.append(response)
response=brief_prompt_gen(LETTERS["L003"],extract_fields(LETTERS["L003"]))
BRIEF_PROMPT.append(response)
response=brief_prompt_gen(LETTERS["L004"],extract_fields(LETTERS["L004"]))
BRIEF_PROMPT.append(response)
response=brief_prompt_gen(LETTERS["L005"],extract_fields(LETTERS["L005"]))
BRIEF_PROMPT.append(response)
response=brief_prompt_gen(LETTERS["L006"],extract_fields(LETTERS["L006"]))
print("Sixth Letter")
print(response)
BRIEF_PROMPT.append(response)

First Letter
**Strengths**  
- Established business with 12 years of operation at a high‑traffic market (Makola).  
- Consistent monthly profit of GHS 900, indicating cash‑flow stability.  
- Demonstrated savings discipline: GHS 2,500 saved via susu scheme over 2 years with no missed contributions.  
- Clear purpose for loan (purchase of deep freezer) that can expand product line into higher‑margin frozen foods.  
- Repayment capacity appears viable: proposed repayment of GHS 450/month is 50 % of current profit, leaving margin for operating expenses.  
- Collateral/guarantor available – sister (teacher) willing to guarantee the loan.  

+-+  

**Risks / Red Flags**  
- Repayment amount (GHS 450) equals 50 % of current profit, leaving limited buffer for unexpected expenses or market downturns.  
- No financial statements or cash‑flow projections provided to validate future profit after adding frozen‑food line.  
- Reliance on a personal guarantor rather than tangible collateral; guarant

In [10]:
print(BRIEF_PROMPT[2])
print("Compare")
print(BRIEF_PROMPT[5])

**Strengths**  
- Consistent profitability: average monthly profit of GHS 2,800 over the past 18 months.  
- Recent strong sales: GHS 22,000 revenue in December 2023, indicating seasonal demand.  
- Clear purpose of loan (purchase of equipment and inventory) that directly supports business growth.  
- Collateral/guarantor available: GHS 5,000 fixed deposit can be pledged.  
- Small, manageable repayment request (GHS 1,100/month) relative to monthly profit.  
- Existing business registration (BN‑2019‑4482) and operational track record.  

+-+  

**Risks / Red Flags**  
- Repayment coverage ratio is modest: repayment ≈ 39 % of average monthly profit, leaving limited buffer for cash‑flow fluctuations.  
- Reliance on a single seasonal peak (Christmas) for revenue; off‑season cash flow may be tighter.  
- Collateral value (GHS 5,000) covers only ~33 % of the loan amount, leaving a sizable unsecured portion.  
- Limited staffing (3 apprentices) may constrain capacity to scale production qui

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** The system's briefs were accurate; it identified the right strengths in both cases and even had more strengths for L003 than L006, showing its superior strength. Forbidding the model from having a direct decision made was because it does not understand compassion and empathy which are human emotions; it also raises ethical concerns since a machine should not be responsible for something that feeds humans(money). Humans also deserve to hear their responses from humans,since they wrote to humans.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

In [17]:
SUMMARY_PROMPT=[SUMMARY_PROMPT_V1,SUMMARY_PROMPT_V2]
EXTRACT_PROMPT
BRIEF_PROMPT
from pprint import pformat
with open("data.py","w") as f:
  f.write(f"SUMMARY_PROMPT={pformat(SUMMARY_PROMPT)}\n")
  f.write(f"EXTRACT_PROMPT={pformat(EXTRACT_PROMPT)}\n")
  f.write(f"BRIEF_PROMPT={pformat(BRIEF_PROMPT)}\n")

In [19]:
from google.colab import files
files.download("data.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [13]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
indices=[0,2,5]
table=pd.DataFrame()
for i in indices:
  fieldsForLetters["applicant_name"]=fieldsForLetters["applicant_name"].apply(
      lambda x:x.lower() if isinstance(x,str) else x
  )
  fieldsForLetters["purpose"]=fieldsForLetters["purpose"].apply(
      lambda x:x.lower() if isinstance(x,str) else x
  )
  GOLD[f"L00{i+1}"]["applicant_name"]=GOLD[f"L00{i+1}"]["applicant_name"].lower()
  GOLD[f"L00{i+1}"]["purpose"]=GOLD[f"L00{i+1}"]["purpose"].lower()
  fieldsForLetters[f"{i+1}applicant_name_correct"]=fieldsForLetters.loc[i,"applicant_name"]==GOLD[f"L00{indices[0]+1}"]["applicant_name"]
  table[f"{i+1}applicant_name_correct"]=fieldsForLetters[f"{i+1}applicant_name_correct"]
  fieldsForLetters[f"{i+1}amount_ghs_correct"]=fieldsForLetters.loc[i,"amount_ghs"]==GOLD[f"L00{indices[0]+1}"]["amount_ghs"]
  table[f"{i+1}amount_ghs_correct"]=fieldsForLetters[f"{i+1}amount_ghs_correct"]
  fieldsForLetters[f"{i+1}purpose_correct"]=fieldsForLetters.loc[i,"purpose"]==GOLD[f"L00{indices[0]+1}"]["purpose"]
  table[f"{i+1}purpose_correct"]=fieldsForLetters[f"{i+1}purpose_correct"]
  fieldsForLetters[f"{i+1}monthly_profit_ghs_correct"]=fieldsForLetters.loc[i,"monthly_profit_ghs"]==GOLD[f"L00{indices[0]+1}"]["monthly_profit_ghs"]
  table[f"{i+1}monthly_profit_ghs_correct"]=fieldsForLetters[f"{i+1}monthly_profit_ghs_correct"]
  fieldsForLetters[f"{i+1}has_collateral_or_guarantor_correct"]=fieldsForLetters.loc[i,"has_collateral_or_guarantor"]==GOLD[f"L00{indices[0]+1}"]["has_collateral_or_guarantor"]
  table[f"{i+1}has_collateral_or_guarantor_correct"]=fieldsForLetters[f"{i+1}has_collateral_or_guarantor_correct"]
  fieldsForLetters[f"{i+1}repayment_months_correct"]=fieldsForLetters.loc[i,"repayment_months"]==GOLD[f"L00{indices[0]+1}"]["repayment_months"]
  table[f"{i+1}repayment_months_correct"]=fieldsForLetters[f"{i+1}repayment_months_correct"]
print(table)
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

   1applicant_name_correct  1amount_ghs_correct  1purpose_correct  \
0                     True                 True             False   
1                     True                 True             False   
2                     True                 True             False   
3                     True                 True             False   
4                     True                 True             False   
5                     True                 True             False   

   1monthly_profit_ghs_correct  1has_collateral_or_guarantor_correct  \
0                         True                                  True   
1                         True                                  True   
2                         True                                  True   
3                         True                                  True   
4                         True                                  True   
5                         True                                  True   

   1repayme

### Part 4.2 — Reliability: is the system consistent?

In [14]:
from numpy import unique_values
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
fields_run=[]
fields_run.append(extract_fields(LETTERS['L004'],temperature=0))
fields_run.append(extract_fields(LETTERS['L004'],temperature=0))
fields_run.append(extract_fields(LETTERS['L004'],temperature=0))
fields_run.append(extract_fields(LETTERS['L004'],temperature=0))
fields_run.append(extract_fields(LETTERS['L004'],temperature=0))
fields_run.append(extract_fields(LETTERS['L004'],temperature=1))
fields_run.append(extract_fields(LETTERS['L004'],temperature=1))
fields_run.append(extract_fields(LETTERS['L004'],temperature=1))
fields_run.append(extract_fields(LETTERS['L004'],temperature=1))
fields_run.append(extract_fields(LETTERS['L004'],temperature=1))
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
validjsons=0
count=0
unique_values=set()
for i in fields_run:
  count+=1
  unique_values.add(json.dumps(i,sort_keys=True))
  try:
    print(json.dumps(i,sort_keys=True))
    validjsons+=1
    if count<5:
      print(f"Temperature of 0 on the {count}th run is valid")
    else:
      print(f"Temperature of 1.0 on the {count}th run is valid")
  except:
    print("Not a valid JSON")
print(f"{validjsons} JSONs were valid.")
print(f"{9-len(unique_values)} JSONs were repeated.")

{"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and purchase 500 new layers", "repayment_months": 18}
Temperature of 0 on the 1th run is valid
{"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and purchase 500 new layers", "repayment_months": 18}
Temperature of 0 on the 2th run is valid
{"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and purchase 500 new layers", "repayment_months": 18}
Temperature of 0 on the 3th run is valid
{"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and purchase 500 new layers", "repayment_months": 18}
Temperature of 0 on the 4th run is valid
{"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guaranto

### Part 4.3 — Hallucination probing

In [16]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
SUMMARY_PROMPT_V1=f"""Summarize this letter acutely:
{LETTERS["L006"]}
What did Kojo say he wanted to eat?
"""
SUMMARY_PROMPT_V2=[system_prompt,SUMMARY_PROMPT_V1,response]
response,usage=ask_llm(SUMMARY_PROMPT_V2[1],SUMMARY_PROMPT_V2[0])
print(response)
#Test 2
extract_fields("The Bible says that Jesus is the way the truth and the life",temperature=0)
# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Summary of the letter**

- **Sender:** Kofi, a 22‑year‑old applicant who found your advertisement.  
- **Requested loan amount:** GHS 50,000.  
- **Intended uses:**  
  1. Launch a car‑washing business.  
  2. Open a provision (general‑goods) shop.  
  3. Import mobile phones from Dubai.  
- **Business status:** None of the ventures have started yet; the plans are still conceptual.  
- **Repayment plan:** Full repayment within one year, contingent on the businesses becoming profitable.  
- **Collateral:** None offered.  
- **Personal assurance:** Kofi emphasizes his energy, business mindset (as attested by friends), and personal trustworthiness.

**Regarding Kojo’s request**

The letter you provided does not contain any reference to a person named Kojo or to any food preference. Therefore, I cannot supply that information. If you have a separate communication that includes Kojo’s comment, please share it and I’ll be happy to help.


{'applicant_name': '',
 'amount_ghs': 0,
 'purpose': '',
 'monthly_profit_ghs': None,
 'has_collateral_or_guarantor': False,
 'repayment_months': None}

In [ ]:
"""
Outputs:
**Summary of the letter**

- **Sender:** Kofi, a 22‑year‑old applicant who found your advertisement.
- **Requested loan amount:** GHS 50,000.
- **Intended uses:**
  1. Launch a car‑washing business.
  2. Open a provision (general‑goods) shop.
  3. Import mobile phones from Dubai.
- **Business status:** None of the ventures have started yet; the plans are still conceptual.
- **Repayment plan:** Full repayment within one year, contingent on the businesses becoming profitable.
- **Collateral:** None offered.
- **Personal assurance:** Kofi emphasizes his energy, business mindset (as attested by friends), and personal trustworthiness.

**Regarding Kojo’s request**

The letter you provided does not contain any reference to a person named Kojo or to any food preference. Therefore, I cannot supply that information. If you have a separate communication that includes Kojo’s comment, please share it and I’ll be happy to help.
Result:PASS

Extraction output:
{'applicant_name': '',
 'amount_ghs': 0,
 'purpose': '',
 'monthly_profit_ghs': None,
 'has_collateral_or_guarantor': False,
 'repayment_months': None}
 Result:PASS(It was supposed to print null/none and not invent data)
"""

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** The extraction accuracy was quite accurate overall; the model obtained all their names and information explicitly mentioned, but for monthly salary it didn't get some for all. Some of the participants did not have a contractral work agreement, so I think that made that the hardest field to extract. The reliability experiment showed that temperature makes responses unique and gives it more of a human color; production systems therefore employ such devices to make responses as natural and human as possible. For my hallucination probing, my model did not hallucinate; it obeyed the prompts given to it and did not make responses up.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** Someone like the author of L005 could be unfairly harmed. They are a joint company, and because they didn't offer any assurance of making money the model could reject them even though they mentioned they have joint collateral saved up. The AI agent would not consider desperation and emotional outbursts, and that amongst other related sentiments is why AI is frowned upon in decision making. They also didn't use the best English; other than that, personal data being sent to a API in another country would allow potential leakage of personal data and raises privacy concerns. Deploying this at a real microfinance institution would require some form of security measures. Some safeguards I could build around this system would be leaving windows for human intervention/supervision and observation before the final decision is forwarded. I would also ensure that the model output is frequently checked, so that the model does not do any faulty work at any time.

#THIS IS THE FINAL INCREMENTAL COMMIT

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?<br>
   It is different because iterating on a prompt would require much more deep learning for the model to provide the sufficient response, which isn't even a fixed response; it is however similar because both involve some form of fine-tuning.
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?<br>
   I would not trust it to run unattended; this was because it processed informal requests as similarly as formal ones, and this is not reflective of the effort put into the requests. It gave equal attention to Kofi and Efua, and that should not be the case. One has more experience, has more on-site demand and is more professional.
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?<br>
   Looking at some estimates, it would take around 514,000 tokens to process that many applications. Providers would therefore have to use a LLM that uses less tokens or put a limit on how many applications are processed monthly.
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?<br>
   For this task, calling an API beats training my own because developing a LLM from scratch would require too much training data and too much storage for its final weights. This would not beat training my own however, for cases that can be sufficiently run locally, since delay is more likely when running with an API.

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.